In [1]:
from keras.utils import to_categorical
from keras_preprocessing.image import load_img
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import os
import pandas as pd
import numpy as np

In [2]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

In [3]:
def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir,label)):
            image_paths.append(os.path.join(dir,label,imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths,labels

In [4]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
print(train)

                                image     label
0            images/train\angry\0.jpg     angry
1            images/train\angry\1.jpg     angry
2           images/train\angry\10.jpg     angry
3        images/train\angry\10002.jpg     angry
4        images/train\angry\10016.jpg     angry
...                               ...       ...
28816  images/train\surprise\9969.jpg  surprise
28817  images/train\surprise\9985.jpg  surprise
28818  images/train\surprise\9990.jpg  surprise
28819  images/train\surprise\9992.jpg  surprise
28820  images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [6]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [7]:
print(test)
print(test['image'])

                              image     label
0       images/test\angry\10052.jpg     angry
1       images/test\angry\10065.jpg     angry
2       images/test\angry\10079.jpg     angry
3       images/test\angry\10095.jpg     angry
4       images/test\angry\10121.jpg     angry
...                             ...       ...
7061  images/test\surprise\9806.jpg  surprise
7062  images/test\surprise\9830.jpg  surprise
7063  images/test\surprise\9853.jpg  surprise
7064  images/test\surprise\9878.jpg  surprise
7065   images/test\surprise\993.jpg  surprise

[7066 rows x 2 columns]
0         images/test\angry\10052.jpg
1         images/test\angry\10065.jpg
2         images/test\angry\10079.jpg
3         images/test\angry\10095.jpg
4         images/test\angry\10121.jpg
                    ...              
7061    images/test\surprise\9806.jpg
7062    images/test\surprise\9830.jpg
7063    images/test\surprise\9853.jpg
7064    images/test\surprise\9878.jpg
7065     images/test\surprise\993.jpg
Name:

In [8]:
from tqdm import tqdm
import numpy as np
from tensorflow.keras.preprocessing.image import load_img

def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode="grayscale")
        img = np.array(img)
        features.append(img)

    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)
    return features

In [9]:
train_features = extract_features(train['image']) 

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28821/28821 [00:21<00:00, 1345.20it/s]


In [10]:
test_features = extract_features(test['image'])

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7066/7066 [00:05<00:00, 1243.87it/s]


In [11]:
x_train = train_features/255.0
x_test = test_features/255.0

In [12]:
from sklearn.preprocessing import LabelEncoder

In [13]:
le = LabelEncoder()
le.fit(train['label'])

LabelEncoder()

In [14]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [15]:
y_train = to_categorical(y_train,num_classes = 7)
y_test = to_categorical(y_test,num_classes = 7)

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D, Input

model = Sequential()

# Input layer
model.add(Input(shape=(48,48,1)))

# Convolutional layers
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.4))

# Fully connected layers
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))

# Output layer
model.add(Dense(7, activation='softmax'))

In [17]:
model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'] )

In [18]:
model.fit(x= x_train,y = y_train, batch_size = 128, epochs = 100, validation_data = (x_test,y_test))

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 299s 1s/step - accuracy: 0.2446 - loss: 1.8241 - val_accuracy: 0.2583 - val_loss: 1.8083
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.2505 - loss: 1.7996 - val_accuracy: 0.2839 - val_loss: 1.7342
Epoch 3/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.2987 - loss: 1.7127 - val_accuracy: 0.3593 - val_loss: 1.6185
Epoch 4/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.3624 - loss: 1.6062 - val_accuracy: 0.4365 - val_loss: 1.4505
Epoch 5/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.4193 - loss: 1.5004 - val_accuracy: 0.4809 - val_loss: 1.3550
Epoch 6/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 345s 2s/step - accuracy: 0.4441 - loss: 1.4447 - val_accuracy: 0.4986 - val_loss: 1.3214
Epoch 7/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.4605 - loss: 1.3959 - val_accuracy: 0.5016 - val_loss: 1.3061
Epoch 8/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.4738 - loss: 1.3627 - 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



226/226 ━━━━━━━━━━━━━━━━━━━━ 4925s 22s/step - accuracy: 0.7276 - loss: 0.7471 - val_accuracy: 0.6363 - val_loss: 1.0333
Epoch 100/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 524s 2s/step - accuracy: 0.7309 - loss: 0.7443 - val_accuracy: 0.6343 - val_loss: 1.0389


In [19]:
model_json = model.to_json()
with open("emotiondetectornew.json",'w') as json_file:
    json_file.write(model_json)
model.save("emotiondetectornew.h5")

In [20]:
from keras.models import model_from_json

In [21]:
json_file = open("emotiondetectornew.json", "r")
model_json = json_file.read()
json_file.close()

model = model_from_json(model_json)
model.load_weights("emotiondetectornew.h5")

In [22]:
label = ['angry','disgust','fear','happy','neutral','sad','surprise']

In [23]:
def ef(image):
    img = load_img(image, color_mode="grayscale")
    feature = np.array(img)
    feature = feature.reshape(1,48,48,1)
    return feature/255.0

In [24]:
image = 'images/train/sad/42.jpg'
print("original image is of sad")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)

original image is of sad
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step
model prediction is  sad
